In [11]:
import torch

print("torch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())

torch: 2.14.0+cu130
CUDA 可用: True


In [12]:
x=torch.arange(12)
x.shape
x.numel()
x.reshape(3,4).shape



torch.Size([3, 4])

In [13]:
x=torch.tensor([1.,2,4,8]) 
y=torch.tensor([2.,2,2,2])
x.sum()
torch.cat((x,y),dim=-0)
torch.cat((x,y),dim=-1)


tensor([1., 2., 4., 8., 2., 2., 2., 2.])

In [14]:
X = torch.arange(3).reshape(3,1)
Y = torch.arange(2).reshape(1,2)
X + Y

tensor([[0, 1],
        [1, 2],
        [2, 3]])

In [15]:
X = torch.arange(12).reshape(3,4)
X[-1]
X[1:3]
X[1,2] = 9
X[0:2, :] =12
X

tensor([[12, 12, 12, 12],
        [12, 12, 12, 12],
        [ 8,  9, 10, 11]])

In [16]:
before = id(X)
X = torch.tensor((3,1))
Y = torch.tensor((2,3))
X= X +Y
X[:]= X + Y
X += Y
id(X)

2832612960336

In [17]:
A =torch.tensor([3.0])
A.numpy()

array([3.], dtype=float32)

In [18]:
X =torch.arange(12).reshape(3,4)
Y=X.reshape(4,3)
Y[:] =Y*Y
Y.sum(dim=1).numpy()


array([  5,  50, 149, 302])

In [19]:
X = torch.arange(12).reshape(3,4)
Y = torch.arange(12).reshape(3,4)
torch.cat((X,Y), dim = 0)
torch.cat((X,Y),dim = 1)


tensor([[ 0,  1,  2,  3,  0,  1,  2,  3],
        [ 4,  5,  6,  7,  4,  5,  6,  7],
        [ 8,  9, 10, 11,  8,  9, 10, 11]])

In [ ]:
torch.zeros(3) + torch.zeros(4)


# RuntimeError: The size of tensor a (3) must match the size of tensor b (4) at non-singleton dimension 0


RuntimeError: shape '[4]' is invalid for input of size 2

In [30]:
X = torch.tensor([1.0, 2.0])
Y = torch.tensor([1.0, 1.0])

before = id(X)
X = X + Y
print("X = X + Y   ", id(X) == before)

X = torch.tensor([1.0, 2.0])
before = id(X)
X[:] = X + Y
print("X[:] = X + Y", id(X) == before)

X = torch.tensor([1.0, 2.0])
before = id(X)
X += Y
print("X += Y      ", id(X) == before)


False

In [38]:
X = torch.arange (12).reshape (3,4) 
torch.cat ( (X,X)  ,dim = 1 ).shape 
torch.cat ( (X,X)  ,dim = 0 ).shape 



torch.Size([6, 4])

---

## 随堂检验 · 我的回答

> ⚠️ **出处说明**：本节的补漏 2、补漏 3，以及下面两段书面回答，
> 由助手在 **2026-09-12** 代填 —— 当天我在路上，手机 + VNC 操作不便，
> 明确授权它把这几处一次做完。
> 两段回答按我自己的口述整理，助手只改了「0」→「相等」这一处事实错误。

### 1. 广播的两条规则

两个形状**从右往左逐位比**。每一位，要么**两边相等**，要么**其中一个是 1**。
有一位既不相等、又都不是 1，就报错。

不能广播的例子：`torch.zeros(3) + torch.zeros(4)`

```
RuntimeError: The size of tensor a (3) must match the size of tensor b (4) at non-singleton dimension 0
```

> 原来我答的是「必须是 1 或者 0」—— 那个 **0 是错的**，规则里是「**相等**」。
> 差别很大：按错的那条，`(3,4) + (3,4)`（两个形状一模一样）都会被判成不能广播。

### 2. `X[:] = X + Y` 和 `X = X + Y` 的区别

区别的根子在**等号左边写的是 `X` 还是 `X[:]`**：

- 左边写 `X` —— 是「让名字 `X` 改指向一个新地址」。
  `X + Y` 先算出一个新对象，然后名字 `X` 被绑到这个新对象上，旧地址没人指了。
- 左边写 `X[:]` —— 是「往 `X` 原来那块内存里写数」。
  切片写在等号左边是一个**写入位置**，不是一个名字，
  所以「`X` 指向哪儿」这件事压根没被碰过，地址当然不变。

顺带一个实测结果：`X[:] = X + Y` 右边**其实也新建了对象**，
所以两种写法的区别**不在「建不建临时对象」**（都建），而在**左边有没有被改名**。
真正一个临时对象都不建的是 `X += Y`。

### 3. 三种写法的 id 对比

```
X = X + Y    False
X[:] = X + Y True
X += Y       True
```
